<a href="https://colab.research.google.com/github/tanishq-soni99/Langraph-career-advisor/blob/main/langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q \
langgraph \
langchain-core \
langchain-community \
gradio \
huggingface_hub

In [6]:
from typing import TypedDict

from langgraph.graph import StateGraph, END

from huggingface_hub import InferenceClient

In [7]:
from google.colab import userdata
from huggingface_hub import InferenceClient

HF_TOKEN = userdata.get("HF_TOKEN")

client = InferenceClient(
    model="meta-llama/Llama-3.1-8B-Instruct",
    token=HF_TOKEN
)

In [8]:
def generate_response(prompt):

    response = client.chat_completion(
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],
        max_tokens=500
    )

    return response.choices[0].message.content

In [9]:
class CareerState(TypedDict):
    skills: str
    experience: str
    goal: str

    analysis: str
    advice: str
    roadmap: str

In [10]:
def analyze_profile(state):

    prompt = f"""
    Analyze this candidate:

    Skills:
    {state['skills']}

    Experience:
    {state['experience']}

    Goal:
    {state['goal']}

    Give strengths and weaknesses.
    """

    result = generate_response(prompt)

    return {
        "analysis": result
    }

In [11]:
def career_advice(state):

    prompt = f"""
    Based on this analysis:

    {state['analysis']}

    Provide career advice.
    """

    result = generate_response(prompt)

    return {
        "advice": result
    }

In [12]:
def learning_roadmap(state):

    prompt = f"""
    Goal:

    {state['goal']}

    Create a 3 month roadmap.
    """

    result = generate_response(prompt)

    return {
        "roadmap": result
    }

In [13]:
builder = StateGraph(CareerState)

builder.add_node("analyze", analyze_profile)
builder.add_node("advice", career_advice)
builder.add_node("roadmap", learning_roadmap)

builder.set_entry_point("analyze")

builder.add_edge("analyze", "advice")
builder.add_edge("advice", "roadmap")
builder.add_edge("roadmap", END)

graph = builder.compile()

In [14]:
def run_agent(skills, experience, goal):

    result = graph.invoke(
        {
            "skills": skills,
            "experience": experience,
            "goal": goal
        }
    )

    return f"""
## Profile Analysis

{result['analysis']}

---

## Career Advice

{result['advice']}

---

## Learning Roadmap

{result['roadmap']}
"""

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=run_agent,
    inputs=[
        gr.Textbox(label="Skills"),
        gr.Textbox(label="Experience"),
        gr.Textbox(label="Career Goal")
    ],
    outputs=gr.Markdown(),
    title="LangGraph Career Advisor"
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ca60820dacbdce2829.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
